Math tutor Specialist

In [11]:
from dotenv import load_dotenv

load_dotenv()

True

In [12]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"

In [31]:
def chat(messages, system=None, temperature=None):
    params = {
        "model": model,
        "messages": messages,
        "max_tokens": 1024,
    }
    
    if system:
        params["system"] = system
        
    if temperature:
        # Claude Sonnet 5 rejects the temperature parameter entirely
        params["temperature"] = temperature
        
    message = client.messages.create(**params)
    return next(block.text for block in message.content if block.type == "text")

In [25]:
def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})
    return messages

def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})
    return messages

In [26]:
messages = []

add_user_message(messages, "How do I solve 5x + 3 = 18 for x?")



[{'role': 'user', 'content': 'How do I solve 5x + 3 = 18 for x?'}]

In [20]:
# Solve without system prompt
from IPython.display import Markdown

answer = chat(messages)

Markdown(answer)

# Solving 5x + 3 = 18

**Step 1: Isolate the term with x**

Subtract 3 from both sides:
$$5x + 3 - 3 = 18 - 3$$
$$5x = 15$$

**Step 2: Solve for x**

Divide both sides by 5:
$$\frac{5x}{5} = \frac{15}{5}$$
$$x = 3$$

**Check your answer:**
Substitute x = 3 back into the original equation:
$$5(3) + 3 = 15 + 3 = 18 ✓$$

So **x = 3**.

In [27]:
# Solve with system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
answer_with_system_prompt = chat(messages, system=system)

Markdown(answer_with_system_prompt)

Great problem to practice with! Let's work through it step by step, but I'll guide you rather than just give you the answer.

First, let's look at the equation: 5x + 3 = 18

**Step 1:** Your goal is to get the term with *x* by itself on one side. Right now, you have "+ 3" on the left side with the 5x. 

What operation would you do to *both sides* of the equation to get rid of that +3 on the left?

Try that and tell me what equation you get.

In [21]:
messages = []

add_user_message(
    messages, 
    "Write a python function that checks a string for duplicate characters and returns True if there are duplicates, otherwise False."
)

answer = chat(messages)

answer

'## Function to Check for Duplicate Characters\n\n```python\ndef has_duplicates(s):\n    """\n    Checks if a string contains duplicate characters.\n    \n    Args:\n        s (str): The input string to check\n        \n    Returns:\n        bool: True if duplicates exist, False otherwise\n    """\n    return len(s) != len(set(s))\n```\n\n### How it works:\n- `set(s)` removes duplicate characters from the string, keeping only unique ones\n- If the length of the set is **less than** the original string\'s length, duplicates exist\n- If they\'re **equal**, all characters are unique\n\n### Example usage:\n\n```python\nprint(has_duplicates("hello"))      # True  (l appears twice)\nprint(has_duplicates("world"))      # False (all unique)\nprint(has_duplicates("python"))     # False (all unique)\nprint(has_duplicates("aabbcc"))     # True  (all duplicated)\nprint(has_duplicates(""))           # False (empty string)\n```\n\n---\n\n### Alternative: Case-Insensitive & Ignoring Spaces\n\nIf you 

In [28]:
messages = []
system = """
You are a helpful assistant that writes Python code.
You will be given a prompt and you should write a Python function that fulfills the requirements of the prompt.
You should give as concise an answer as possible, and only write the code for the function.
"""

add_user_message(
    messages, 
    "Write a python function that checks a string for duplicate characters and returns True if there are duplicates, otherwise False."
)

answer = chat(messages, system=system)

Markdown(answer)

```python
def has_duplicates(s):
    return len(s) != len(set(s))
```

In [ ]:
messages = []

print("Movie ideas with different temperature settings")

add_user_message(
    messages, 
    "Generate catchy movie ideas"
)

print("Low temperature - more predictable")

# Low temperature - more predictable
answer = chat(messages, temperature=0.0)

display(Markdown(answer))

print("High temperature - more creative")

# High temperature - more creative  
answer = chat(messages, temperature=1.0)

display(Markdown(answer))

passing temperate as parameter to create function may be deprecated error


FIX:

The root cause: chat() defaulted temperature=1.0 and always included it in the request params, but Claude Sonnet 5 rejects the temperature parameter entirely (deprecated for this model) unless it's actually needed. Now temperature defaults to None and is only added to params when explicitly passed — which still works for the later cell that calls chat(messages, temperature=0.0) / temperature=1.0, but no longer breaks the earlier calls that omit it.

## Streaming methods comparison

| Method | What it is | When to choose it |
|---|---|---|
| **`client.messages.stream()`** (context manager) | High-level helper: accumulates state, exposes `stream.text_stream` for incremental text and `stream.get_final_message()` for the complete `Message` at the end | **Default choice.** Building a chat UI, need incremental text display, and/or want the complete message afterward (usage stats, tool_use blocks) without manually reassembling it |
| **`stream=True` on `messages.create()`** (raw event iterator) | Low-level: yields raw `RawMessageStreamEvent` objects (`message_start`, `content_block_delta`, `message_delta`, etc.) with no accumulation | You only need to react to specific event types (e.g. just track token usage from `message_delta`, or build a custom accumulator) and want to avoid the memory/overhead of the high-level wrapper |
| **Non-streaming `messages.create()`** (no streaming at all) | One blocking call, returns complete `Message` | Short outputs (`max_tokens` roughly ≤16K), no need for progressive display, simplest code path |
| **Tool Runner with `stream=True`** | Combines the agentic tool-use loop with streaming — each loop iteration yields a stream you consume event-by-event | Agentic loops where you want to stream Claude's text *and* have the SDK handle the tool-call round-trips automatically |

### Decision guide

- **Any output that could be long or slow** (agentic tasks, `max_tokens` > ~16K, or unpredictable latency) → **always stream**, regardless of UI needs. Non-streaming requests above ~16K output risk hitting SDK HTTP timeouts (idle connections drop after ~10 min); the Python SDK raises `ValueError` if it estimates a non-streaming call will exceed that.
- **Need to show tokens as they arrive to a user** → `client.messages.stream()` + `stream.text_stream`.
- **Need the full `Message` object afterward** (usage, stop_reason, tool_use blocks) even though you streamed → still use `client.messages.stream()`, call `stream.get_final_message()` at the end.
- **Building a custom low-level pipeline** → raw `stream=True`.
- **No latency/length concerns, just need the answer** → non-streaming `create()`, simplest code.
- **Agentic tool-use + streaming text** → Tool Runner with `stream=True`.

**Bottom line:** for almost all interactive or long-output use cases, `client.messages.stream()` is the right default — reach for raw `stream=True` only when you specifically want to skip the SDK's accumulation logic.
